In [1]:
import pandas as pd
import numpy as np
import json
import os

In [2]:
train = pd.read_csv(r"C:\Users\DELL\Coding\Projects\Customer-Churn-Prediction\data\raw\cell2celltrain.csv")
holdout = pd.read_csv(r"C:\Users\DELL\Coding\Projects\Customer-Churn-Prediction\data\raw\cell2cellholdout.csv")
telco = pd.read_excel(r"C:\Users\DELL\Coding\Projects\Customer-Churn-Prediction\data\raw\Telco_customer_churn.xlsx")

In [3]:
print("Train Shape:", train.shape)
print("Holdout Shape:", holdout.shape)
print("Telco Shape:", telco.shape)

Train Shape: (51047, 58)
Holdout Shape: (20000, 58)
Telco Shape: (7043, 33)


In [4]:
print("Holdout Churn unique:", holdout['Churn'].unique())
print("All labels missing:", holdout['Churn'].isna().all())

Holdout Churn unique: [nan]
All labels missing: True


In [5]:
print(train['Churn'].value_counts())

print("\nPercentage:")
print(train['Churn'].value_counts(normalize=True))

Churn
No     36336
Yes    14711
Name: count, dtype: int64

Percentage:
Churn
No     0.711815
Yes    0.288185
Name: proportion, dtype: float64


In [6]:
print(train.duplicated().sum())
print(train.dtypes.value_counts())

0
float64    26
object     23
int64       9
Name: count, dtype: int64


In [7]:
train.head()

,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,...,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
0,3000002,Yes,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,...,0,4,No,0,30,Yes,1-Highest,Suburban,Professional,No
1,3000010,Yes,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,...,0,5,No,0,30,No,4-Medium,Suburban,Professional,Yes
2,3000014,No,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,...,0,6,No,0,Unknown,No,3-Good,Town,Crafts,Yes
3,3000022,No,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,...,0,6,No,0,10,No,4-Medium,Other,Other,No
4,3000026,Yes,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,...,0,9,No,1,10,No,1-Highest,Other,Professional,Yes


In [8]:
telco.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [9]:
train_num = train.select_dtypes(include=np.number).copy()

train_num['Churn_num'] = (
    train['Churn'] == 'Yes'
).astype(int)

corr = (
    train_num
    .corrwith(train_num['Churn_num'])
    .abs()
    .drop('Churn_num')
    .sort_values(ascending=False)
)

In [10]:
print(corr.head(25))

CurrentEquipmentDays         0.103688
RetentionCalls               0.065301
TotalRecurringCharge         0.061296
MonthlyMinutes               0.050166
CustomerID                   0.041558
OffPeakCallsInOut            0.040784
HandsetModels                0.040048
PeakCallsInOut               0.040029
ReceivedCalls                0.037453
CustomerCareCalls            0.035510
RetentionOffersAccepted      0.034953
UniqueSubs                   0.034522
InboundCalls                 0.034232
PercChangeMinutes            0.034202
Handsets                     0.032845
OutboundCalls                0.032276
AgeHH1                       0.029741
UnansweredCalls              0.029350
CallWaitingCalls             0.025653
ThreewayCalls                0.023854
DirectorAssistedCalls        0.019516
MonthsInService              0.018703
AgeHH2                       0.018576
AdjustmentsToCreditRating    0.017056
OverageMinutes               0.016461
dtype: float64


In [11]:
print(corr.tail(10))

ActiveSubs                   0.015515
DroppedCalls                 0.015397
DroppedBlockedCalls          0.013266
IncomeGroup                  0.012709
MonthlyRevenue               0.011987
PercChangeRevenues           0.011117
RoamingCalls                 0.010882
ReferralsMadeBySubscriber    0.010686
BlockedCalls                 0.005530
CallForwardingCalls          0.001449
dtype: float64


In [12]:
CELL2CELL_DROP = [
    # Identifier
    'CustomerID',
    # Exact opposite of NewCellphoneUser
    'NotNewCellphoneUser',
    # Very low correlation
    'CallForwardingCalls',
    # Very low correlation
    'ReferralsMadeBySubscriber',
    # Very low correlation
    'BlockedCalls',
]

In [13]:
print("\nServiceArea unique values:")
print(train['ServiceArea'].nunique())


ServiceArea unique values:
747


In [14]:
print(
    sorted(
        train['HandsetPrice']
        .dropna()
        .unique()
    )
)


['10', '100', '130', '150', '180', '200', '240', '250', '30', '300', '40', '400', '500', '60', '80', 'Unknown']


In [15]:
print(
    "MonthlyRevenue min:",
    train['MonthlyRevenue'].min()
)
print(
    "CurrentEquipmentDays min:",
    train['CurrentEquipmentDays'].min()
)
print(
    "CustomerCareCalls max:",
    train['CustomerCareCalls'].max()
)
print(
    "RetentionCalls max:",
    train['RetentionCalls'].max()
)
print(
    "RetentionOffersAccepted max:",
    train['RetentionOffersAccepted'].max()
)
print(
    "MonthsInService range:",
    train['MonthsInService'].min(),
    train['MonthsInService'].max()
)


MonthlyRevenue min: -6.17
CurrentEquipmentDays min: -5.0
CustomerCareCalls max: 327.3
RetentionCalls max: 4
RetentionOffersAccepted max: 3
MonthsInService range: 6 61


In [16]:
missing = train.isnull().sum()
print(
    missing[
        missing > 0
    ]
)

MonthlyRevenue           156
MonthlyMinutes           156
TotalRecurringCharge     156
DirectorAssistedCalls    156
OverageMinutes           156
RoamingCalls             156
PercChangeMinutes        367
PercChangeRevenues       367
ServiceArea               24
Handsets                   1
HandsetModels              1
CurrentEquipmentDays       1
AgeHH1                   909
AgeHH2                   909
dtype: int64


In [17]:
print(
    "Senior Citizen dtype:",
    telco['Senior Citizen'].dtype
)

print(
    "Total Charges NaN after conversion:",
    pd.to_numeric(
        telco['Total Charges'],
        errors='coerce'
    ).isna().sum()
)

Senior Citizen dtype: object
Total Charges NaN after conversion: 11


In [18]:
audit = {
    'cell2cell_train_shape':
        list(train.shape),
    'cell2cell_holdout_shape':
        list(holdout.shape),
    'holdout_has_labels':
        False,
    'telco_shape':
        list(telco.shape),
    'cell2cell_churn_rate':
        float(
            (train['Churn'] == 'Yes').mean()
        ),
    'telco_churn_rate':
        float(
            telco['Churn Value'].mean()
        ),
    'cell2cell_drop_cols':
        CELL2CELL_DROP,
    'service_area_unique':
        int(
            train['ServiceArea']
            .nunique()
        ),
}
print(audit)
with open(
    'C:/Users/DELL/Coding/Projects/Customer-Churn-Prediction/reports/dataset_audit.json',
    'w'
) as f:

    json.dump(
        audit,
        f,
        indent=2
    )

print("\nAudit saved successfully.")

{'cell2cell_train_shape': [51047, 58], 'cell2cell_holdout_shape': [20000, 58], 'holdout_has_labels': False, 'telco_shape': [7043, 33], 'cell2cell_churn_rate': 0.2881853977706819, 'telco_churn_rate': 0.2653698707936959, 'cell2cell_drop_cols': ['CustomerID', 'NotNewCellphoneUser', 'CallForwardingCalls', 'ReferralsMadeBySubscriber', 'BlockedCalls'], 'service_area_unique': 747}

Audit saved successfully.


In [ ]:
with open("C:/Users/DELL/Coding/Projects/Customer-Churn-Prediction/reports/dataset_audit.json", "r") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

{
  "cell2cell_train_shape": [
    51047,
    58
  ],
  "cell2cell_holdout_shape": [
    20000,
    58
  ],
  "holdout_has_labels": false,
  "telco_shape": [
    7043,
    33
  ],
  "cell2cell_churn_rate": 0.2881853977706819,
  "telco_churn_rate": 0.2653698707936959,
  "cell2cell_drop_cols": [
    "CustomerID",
    "NotNewCellphoneUser",
    "CallForwardingCalls",
    "ReferralsMadeBySubscriber",
    "BlockedCalls"
  ],
  "service_area_unique": 747
}
